# Day 8 Exercises: Backpropagation

Student version with gaps.

## Context

This lab opens the training black box from Day 7. You will manually implement backpropagation twice: first for a shallow one-hidden-layer ReLU network, then for a deeper two-hidden-layer ReLU network. After that, you will compare your gradients with JAX's reverse-mode backpropagation and inspect why reverse mode is efficient for scalar neural-network losses.

The networks and batch are deliberately small. The point is not to obtain an impressive fit, but to see which values must be cached, how local derivative rules compose, and what JAX automates for us.

> #### Instructions
>
> Run the setup cells first. Complete the exercise-local implementation cells, then run the three exercises. Use the shape checks, gradient comparisons, and efficiency diagnostics to answer the open questions.
> The exercise cells use the functions from that block, so run each exercise after completing the relevant implementation functions.


## Learning Goals

The lecture describes backpropagation as reverse accumulation through a computational graph. This notebook turns that description into batch-oriented code for a shallow and a deep ReLU network.

By the end of the lab you should be able to:

- implement manual backpropagation for a shallow one-hidden-layer network;
- use manual gradients in gradient descent for a mean-squared-error loss;
- extend the same local derivative rules to a deeper two-hidden-layer network;
- compare manual gradients with JAX reverse-mode gradients from `jax.grad`;
- explain what JAX automates compared with a manual backpropagation implementation;
- compare reverse-mode backpropagation with forward-mode automatic differentiation for a scalar loss.


## Shared Backpropagation Workflow

### Notebook convention
Blocks labelled **Instructions** tell you what to do. Blocks labelled **Open questions** are the questions to answer after running the diagnostics.

### Structure
Both networks use the same data batch, activation function, MSE loss convention, local backward rules, and diagnostic helpers. The shallow network is the minimal case. The deep network shows that backpropagation is the same local mechanism repeated through more layers.

### What stays fixed?
The data batch, parameter dictionaries, mean-squared-error convention, local affine and ReLU backward rules, JAX reference gradients, gradient-comparison tables, and efficiency helpers are shared.

### What changes between exercises?
Exercise 1 asks you to implement manual backpropagation for the shallow and deep models. Exercise 2 compares both manual implementations with JAX backpropagation. Exercise 3 compares reverse-mode backpropagation with forward-mode AD from an efficiency viewpoint.


## Setup

The shared data, fixed networks, JAX references, shape checks, gradient-descent diagnostics, and forward/reverse-mode efficiency helpers live next to this notebook in `day8_helpers.py`. The notebook imports those provided pieces here; the implementation tasks remain inside the exercise sections below.

You may open `day8_helpers.py` if you want to inspect the provided infrastructure. This is optional: the core lab does not require reading or editing the helper file.

> #### Instructions
>
> Run this cell before starting the exercises. You do not need to edit the helper module during the core lab.


In [ ]:
from pathlib import Path
import sys

# Make the local helper module importable when running from the day folder,
# the notebooks folder, the quarto folder, or the course root.
for helper_dir in [
    Path.cwd(),
    Path.cwd() / "day8",
    Path.cwd() / "notebooks" / "day8",
    Path.cwd() / "quarto" / "notebooks" / "day8",
]:
    if (helper_dir / "day8_helpers.py").exists():
        sys.path.insert(0, str(helper_dir))
        break

from day8_helpers import *  # Provided teaching infrastructure for this lab.


## Data and Parameter Inspection

This cell instantiates the two fixed networks and the tiny training batch. Part a uses the shallow network; part b uses the deep network. Both see the same batch, so differences in the backward pass come from graph depth, not from different data.

The printed shapes are meant to be used while reading the formulas in Exercise 1. For example, `W1` maps the input column to hidden activations, and each bias has one entry per hidden unit.

> #### Instructions
>
> Run the cell and inspect the parameter shapes before starting Exercise 1.


In [ ]:
shallow_params = init_shallow_network()
deep_params = init_deep_network()
x_batch, y_batch = make_tiny_batch()

print_parameter_shapes("Shallow network parameters", shallow_params, SHALLOW_ORDER)
print_parameter_shapes("Deep network parameters", deep_params, DEEP_ORDER)
plot_batch_and_predictions(x_batch, y_batch)


## Exercise 1: Manual Backpropagation and Gradient Descent

**Part a** implements the neural network and its MSE gradient explicitly for the shallow network

$$
x \mapsto h_1=\sigma(W_1x+b_1), \qquad \hat y=W_2h_1+b_2.
$$

**Part b** repeats the same idea for the deep network

$$
x \mapsto h_1=\sigma(W_1x+b_1), \qquad h_2=\sigma(W_2h_1+b_2), \qquad \hat y=W_3h_2+b_3.
$$

After checking the one-pass gradients, the provided diagnostics use those manual gradients in gradient descent to reduce the same mean-squared-error loss.

> #### Instructions
>
> 1. Complete the activation/loss helper cell and the shallow forward pass.
> 2. Complete the shallow MSE-gradient code.
> 3. Complete the deep forward pass.
> 4. Complete the deep MSE-gradient code.
> 5. Run the diagnostics cell and inspect the cache shapes, gradient shapes, loss histories, and final fitted values.

> #### Further testing
>
> After the main diagnostic run, change one small setting if time permits and compare the table or plot before answering.
>
> #### Open questions
>
> 1. Which gradient formulas are reused in part a and part b?
> 2. Which additional cached values are needed for the deep network?
> 3. How do the MSE loss histories change during gradient descent for the shallow and deep networks?


In [ ]:
# Exercise 1 activation and loss tasks

def relu(z):
    # ======================================================================
    # STUDENT TASK: ReLU activation
    # Replace the placeholder line below with an elementwise ReLU.
    # Hint: Use an elementwise JAX expression such as jnp.where or jnp.maximum, not np.where or np.maximum.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the JAX ReLU activation for Exercise 1.")


def relu_prime(z):
    # ======================================================================
    # STUDENT TASK: ReLU derivative
    # Replace the placeholder line below with the elementwise derivative.
    # Hint: Use JAX-compatible logic, such as jnp.where or a JAX boolean mask with astype; avoid NumPy masks here.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the JAX ReLU derivative for Exercise 1.")


def mse_loss_and_output_adjoint(y_hat, y):
    """Return mean-squared error and d loss / d y_hat."""
    # ======================================================================
    # STUDENT TASK: MSE value and output adjoint
    # Replace the placeholder line below with both the scalar MSE and its
    # derivative with respect to y_hat.
    # Hint: Use JAX reductions such as jnp.mean; do not use np.mean inside code that participates in gradient checks.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the MSE value and output adjoint for Exercise 1.")


> #### Instructions
>
> Complete the shallow-network forward pass. Cache the values that the backward pass will need.


In [ ]:
def shallow_forward_with_cache(params, x):
    """Part a: explicit shallow network forward pass."""
    # ======================================================================
    # STUDENT TASK: shallow forward pass
    # Replace the placeholder line below by the first affine map, ReLU
    # activation, output affine map, cache construction, and return values.
    # Hint: Use JAX array operations (@ and +) together with relu, and store the resulting JAX arrays in the cache.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the shallow forward pass for Exercise 1a.")


> #### Instructions
>
> Complete the shallow-network backward pass. Start from the MSE output adjoint, move through the output affine layer, then through the ReLU and first affine layer.


In [ ]:
def shallow_manual_backward(params, cache, y):
    """Part a: explicit MSE gradient for the shallow network."""
    _, dy_hat = mse_loss_and_output_adjoint(cache["y_hat"], y)

    # ======================================================================
    # STUDENT TASK: shallow backward pass
    # Replace the placeholder line below by the transposed-Jacobian products
    # for the output layer, ReLU, and first affine layer, then return gradients.
    # Hint: Keep the backward pass in JAX array operations, for example @ and jnp.sum; avoid np.* in gradient computations.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the shallow backward pass for Exercise 1a.")


> #### Instructions
>
> Complete the deep-network forward pass. Cache every pre-activation and activation that the backward pass will need.


In [ ]:
def deep_forward_with_cache(params, x):
    """Part b: explicit deep network forward pass."""
    # ======================================================================
    # STUDENT TASK: deep forward pass
    # Replace the placeholder line below by affine -> ReLU -> affine -> ReLU
    # -> affine, cache construction, and return values.
    # Hint: As in the shallow case, use JAX array operations (@, +, relu) and cache JAX arrays for both hidden layers.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the deep forward pass for Exercise 1b.")


> #### Instructions
>
> Complete the deep-network backward pass. Reuse the same local rules as in the shallow network, moving from the output layer back to the first layer.


In [ ]:
def deep_manual_backward(params, cache, y):
    """Part b: explicit MSE gradient for the deep network."""
    _, dy_hat = mse_loss_and_output_adjoint(cache["y_hat"], y)

    # ======================================================================
    # STUDENT TASK: deep backward pass
    # Replace the placeholder line below by the backward sweep through output,
    # second hidden, and first hidden layers, then return gradients.
    # Hint: Use @ for matrix adjoint products and jnp.sum for bias gradients while propagating adjoints through the layers.
    # YOUR CODE HERE
    # ======================================================================
    raise NotImplementedError("Implement the deep backward pass for Exercise 1b.")


> #### Instructions
>
> Choose the short diagnostic training settings and run the shared checks. This is where your forward and backward functions are checked against shapes and then used in manual gradient descent.
>
> A good starting point is `SHALLOW_LEARNING_RATE = 0.05`, `DEEP_LEARNING_RATE = 0.03`, and `N_STEPS = 120`. If the loss increases or becomes `nan`, reduce the learning rate by a factor of 2 to 10. If the loss barely changes, try values from `{0.01, 0.03, 0.05, 0.1}` while keeping `N_STEPS` around 100 to 200. The diagnostics use the full tiny batch, so there is no minibatch-size choice in Exercise 1.


In [ ]:
# ======================================================================
# STUDENT TASK: short manual-training settings
# Define stable learning rates and a modest number of gradient-descent steps
# for the diagnostics.
# Suggested start: SHALLOW_LEARNING_RATE = 0.05, DEEP_LEARNING_RATE = 0.03,
# and N_STEPS = 120. Try learning rates from [0.01, 0.03, 0.05, 0.1].
# Hint: These are plain Python hyperparameters; no JAX array operation is needed for this settings list.
# YOUR CODE HERE
# ======================================================================
raise NotImplementedError("Choose short manual-training settings for Exercise 1.")


> #### Instructions
>
> Run the manual-backpropagation diagnostics after choosing the short training settings.


In [ ]:
manual_backprop_results = run_manual_backpropagation_diagnostics(
    shallow_params,
    deep_params,
    x_batch,
    y_batch,
    shallow_forward_with_cache,
    shallow_manual_backward,
    deep_forward_with_cache,
    deep_manual_backward,
    relu,
    relu_prime,
    mse_loss_and_output_adjoint,
    shallow_learning_rate=SHALLOW_LEARNING_RATE,
    deep_learning_rate=DEEP_LEARNING_RATE,
    steps=N_STEPS,
)
shallow_manual_grads = manual_backprop_results["shallow_grads"]
deep_manual_grads = manual_backprop_results["deep_grads"]


> #### Further testing
>
> If time permits, change only `N_STEPS` or one learning rate and rerun the diagnostic.
>
> #### Open questions
>
> 1. Which cached shallow-network values were essential for the manual backward pass?
> 2. Which extra cache entries did the deep network need?
> 3. How do the shallow and deep MSE histories change during the short manual-gradient run?


## Exercise 2: Compare With JAX Backpropagation

JAX's `jax.grad` applies reverse-mode automatic differentiation to the same scalar loss. In this exercise, JAX is the reference implementation of backpropagation. Your manual gradients should agree with it up to roundoff error.

> #### Instructions
>
> 1. Complete the two short JAX-gradient calls.
> 2. Run the comparison and one-step loss checks.
> 3. Use the maximum absolute errors to judge whether your manual gradients agree with JAX.

> #### Further testing
>
> After the main diagnostic run, change one small setting if time permits and compare the table or plot before answering.
>
> #### Open questions
>
> 1. Which parameter has the largest absolute discrepancy in the shallow comparison? What is its size?
> 2. Which parameter has the largest absolute discrepancy in the deep comparison? What is its size?
> 3. What does JAX automate that you had to write explicitly in Exercise 1?


In [ ]:
# ======================================================================
# STUDENT TASK: JAX reference gradients
# Replace the placeholder line below by reverse-mode gradients from JAX.
# Hint: Use jax.grad on each scalar JAX loss function; pass the existing parameter pytree together with the batch arrays.
# YOUR CODE HERE
# ======================================================================
raise NotImplementedError("Compute the JAX reference gradients for Exercise 2.")


> #### Instructions
>
> Check the shallow manual gradients against JAX reverse-mode gradients, then run the one-step loss sanity check.


In [ ]:
# ======================================================================
# STUDENT TASK: shallow gradient tolerance
# Define a numerical tolerance for the shallow gradient comparison.
# Hint: This tolerance is a plain Python float; choose it from the printed JAX-vs-manual gradient errors.
# YOUR CODE HERE
# ======================================================================
raise NotImplementedError("Choose a shallow-gradient comparison tolerance for Exercise 2.")


> #### Instructions
>
> Check the shallow manual gradients against JAX reverse-mode gradients, then run the one-step loss sanity check.


In [ ]:
print_gradient_comparison(
    "Shallow manual gradients versus JAX backpropagation",
    shallow_manual_grads,
    shallow_reference_grads,
    SHALLOW_ORDER,
)
assert_gradient_agreement(
    shallow_manual_grads,
    shallow_reference_grads,
    SHALLOW_ORDER,
    atol=SHALLOW_GRADIENT_TOLERANCE,
)
print_loss_step(
    "Shallow one-step loss check",
    shallow_reference_loss,
    shallow_params,
    shallow_manual_grads,
    SHALLOW_ORDER,
    x_batch,
    y_batch,
)


> #### Instructions
>
> Repeat the same JAX comparison for the deep network. Use the parameter-by-parameter table to locate any mismatch.


In [ ]:
# ======================================================================
# STUDENT TASK: deep gradient tolerance
# Define a numerical tolerance for the deep gradient comparison.
# Hint: This tolerance is a plain Python float; choose it from the printed JAX-vs-manual gradient errors.
# YOUR CODE HERE
# ======================================================================
raise NotImplementedError("Choose a deep-gradient comparison tolerance for Exercise 2.")


> #### Instructions
>
> Repeat the same JAX comparison for the deep network. Use the parameter-by-parameter table to locate any mismatch.


In [ ]:
print_gradient_comparison(
    "Deep manual gradients versus JAX backpropagation",
    deep_manual_grads,
    deep_reference_grads,
    DEEP_ORDER,
)
assert_gradient_agreement(
    deep_manual_grads,
    deep_reference_grads,
    DEEP_ORDER,
    atol=DEEP_GRADIENT_TOLERANCE,
)
print_loss_step(
    "Deep one-step loss check",
    deep_reference_loss,
    deep_params,
    deep_manual_grads,
    DEEP_ORDER,
    x_batch,
    y_batch,
)


> #### Further testing
>
> If a comparison fails, inspect the largest parameter-wise error before changing tolerances.
>
> #### Open questions
>
> 1. Which gradient entries agree to roundoff level?
> 2. What coding mistakes would produce a shape-correct but numerically wrong gradient?
> 3. What does JAX automate that you wrote explicitly in Exercise 1?


## Exercise 3: Efficiency of Backpropagation Versus Forward-Mode AD

For one scalar loss and many parameters, reverse mode computes the full gradient with one backward sweep after the forward pass. Forward mode naturally computes directional derivatives. To recover the full gradient by forward mode, one needs one direction per parameter.

The sweep-count table is the main diagnostic. The timing table is only illustrative because small JAX timings include overhead and depend on the machine.

> #### Instructions
>
> 1. Complete the short efficiency-table builder.
> 2. Run the diagnostics for hidden widths `2`, `8`, and `32`.
> 3. Compare shallow and deep parameter counts, cached scalar values, and sweep counts.
> 4. Inspect the small timing comparison for repeated JVP-based forward mode versus one reverse-mode gradient.

> #### Further testing
>
> After the main diagnostic run, change one small setting if time permits and compare the table or plot before answering.
>
> #### Open questions
>
> 1. How does the number of forward-mode sweeps scale with the number of parameters?
> 2. Why can reverse mode compute the full parameter gradient with one reverse sweep for a scalar loss?
> 3. What extra memory does reverse mode need, and when could forward mode still be preferable?


In [ ]:
def make_efficiency_table(widths, batch_size=6):
    """Compare full-gradient sweep counts for forward and reverse mode."""
    rows = []
    for width in widths:
        # ======================================================================
        # STUDENT TASK: architecture sweep-count rows
        # Replace the placeholder line below by rows for shallow and deep
        # networks of the current width.
        # Hint: This table is ordinary Python bookkeeping; the JAX-specific work is hidden inside the helper counts.
        # YOUR CODE HERE
        # ======================================================================
        raise NotImplementedError("Build the shallow/deep sweep-count rows for Exercise 3.")
    return rows


> #### Instructions
>
> Choose the hidden widths for the conceptual sweep-count table and a smaller list for the illustrative timing check. The sweep counts are the main result; timings are secondary.
>
> Start with `widths = [2, 8, 32]` and `timing_widths = [2, 4, 8]`. The diagnostic cells use `batch_size=x_batch.shape[0]`, which is 6 for the fixed toy batch. If you experiment with the helper functions after the checks pass, try batch sizes such as 4, 8, or 16 and watch how cached scalars and timing change.


In [ ]:
# ======================================================================
# STUDENT TASK: efficiency diagnostic settings
# Define width lists for the conceptual sweep-count table and the small
# timing check.
# Suggested start: widths = [2, 8, 32] and timing_widths = [2, 4, 8].
# The provided diagnostics use the fixed toy batch size x_batch.shape[0] == 6.
# Hint: These are plain Python lists of widths; the timing helper will handle the JAX computations.
# YOUR CODE HERE
# ======================================================================
raise NotImplementedError("Choose width lists for the Exercise 3 efficiency diagnostics.")


> #### Instructions
>
> Run the sweep-count table and the small timing diagnostic after choosing the width lists.


In [ ]:
efficiency_rows = make_efficiency_table(widths, batch_size=x_batch.shape[0])
check_efficiency_table(efficiency_rows, widths, batch_size=x_batch.shape[0])
print_efficiency_table(efficiency_rows)
plot_efficiency_table(efficiency_rows)

timing_rows = time_gradient_methods(timing_widths, batch_size=x_batch.shape[0])
print_timing_table(timing_rows)


> #### Further testing
>
> If the timing cell is slow on your machine, reduce `timing_widths` and keep the conceptual table unchanged.
>
> #### Open questions
>
> 1. How does the forward-mode sweep count scale with the number of parameters?
> 2. What information does reverse mode need to keep from the forward pass?
> 3. In what kind of derivative task could forward mode be the more natural choice?


## Optional Extension: Input Derivatives as a Bridge to PINNs

PINNs use the same automatic-differentiation machinery, but often differentiate the network output with respect to the input variable as well as the parameters. The next cell computes one input derivative for the deep network.

> #### Instructions
>
> Run the optional cell if you have time. This is a preview of Day 9, not a required PINN implementation.


In [ ]:
def scalar_deep_network_output(x_scalar):
    x = jnp.array([[x_scalar]], dtype=jnp.float64)
    return deep_reference_forward(deep_params, x)[0, 0]

x0 = 0.2
input_derivative = jax.grad(scalar_deep_network_output)(x0)
print(f"d y_hat / d x at x={x0}: {float(input_derivative):.6f}")


## Reflection and Deliverable

> #### Instructions
>
> Before you finish, make sure that all exercise-local implementation gaps have been completed and all assertions pass. Then write short answers to the prompts below.

> #### Reflection questions
>
> 1. In part a, which cached shallow-network values were essential for the manual backward pass?
> 2. In part b, what changed when you extended the manual implementation to the deep network?
> 3. How did the MSE loss change when gradient descent used your manual gradients?
> 4. How close were your manual gradients to JAX backpropagation, and what would a large discrepancy suggest?
> 5. Why is reverse-mode backpropagation usually preferred over forward mode for neural-network training losses?

> #### Optional extensions
>
> - replace ReLU by `tanh` and adjust the activation derivative;
> - add one more hidden layer to the deep network;
> - compare forward finite differences with forward-mode JVPs for one parameter direction;
> - compute `d y_hat / d x` at several input points and plot the result.


> #### Development note
>
> Codex, a generative AI tool, was used to help generate and refine this notebook.
